In order to setup the sqs for s3 bucket we need 2 steps:
1. setup policy for sqs to allow s3 bucket to sendMessage to the queue
2. setup notification configuration to send the notification on an event setup

In [48]:
import boto3

In [49]:
# we should creat a session to share the config so that no need to specify again and again
session = boto3.Session(
    region_name='us-east-1',
    aws_access_key_id='test',
    aws_secret_access_key='test'
)

In [35]:
endpoint = "http://localhost:4566"

In [36]:
sqs = session.client("sqs", endpoint_url=endpoint)

In [37]:
queue_url = sqs.get_queue_url(QueueName="dataset-submission")["QueueUrl"]

In [38]:
# Step 1: setup permission to allow S3 to send notification to SQS

there are 2 types of policy. resource-based policy and identify-based policy (user, group, role)
what is a policy: it contains a set of statements 
Each Satement contains: Effect (allow or not, default no), Princi (allow for who), Action (what action can be perform), Resource (Destination resource) and Condition (optional) 


In [39]:
queue_arn = sqs.get_queue_attributes(
    QueueUrl=queue_url,
    AttributeNames=["QueueArn"]
)['Attributes']['QueueArn']

In [40]:
queue_arn

'arn:aws:sqs:us-east-1:000000000000:dataset-submission'

In [41]:
bucket_name="data-platform-raw"

In [42]:
policy = {
    "Version": "2012-10-17",
    "Statement": [{
        "Effect": "Allow",
        "Principal": "s3.amazonaws.com",
        "Action": "sqs:SendMessage",
        "Resource": queue_arn,
        "Condition": {
            "ArnEquals": {"aws:SourceArn": f"arn:aws:s3:::{bucket_name}"},
            "StringEquals": {"aws:SourceAccount": queue_arn.split(":")[4]}
            }
    },
    ]
}

In [43]:
sqs.set_queue_attributes(QueueUrl=queue_url, Attributes={"Policy": json.dumps(policy)})

{'ResponseMetadata': {'RequestId': '521ac704-318a-478a-b04f-3d7d53f550bb',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'content-type': 'application/x-amz-json-1.0;charset=UTF-8',
   'content-length': '2',
   'date': 'Sun, 15 Nov 2026 22:54:08 GMT',
   'x-amz-id-2': '521ac704-318a-478a-b04f-3d7d53f550bb',
   'x-amz-request-id': '521ac704-318a-478a-b04f-3d7d53f550bb',
   'x-amzn-requestid': '521ac704-318a-478a-b04f-3d7d53f550bb'},
  'RetryAttempts': 0}}

In [45]:
# step 2: setup the notification on s3 bucket side

In [44]:
s3 = session.client("s3", endpoint_url=endpoint)

In [47]:
s3.put_bucket_notification_configuration(
    Bucket=bucket_name,
    NotificationConfiguration= {
        "QueueConfigurations": [{
            "Id": "notification-on-incoming-upload",
            "QueueArn": queue_arn,
            "Events": ["s3:ObjectCreated:*"],
            "Filter": {
                "Key": {
                    "FilterRules": [{"Name": "prefix", "Value": "incoming/"}]
                }
            }
        },
        ]                       
    }
)

{'ResponseMetadata': {'RequestId': '33af6fd6-2e3c-4223-b050-ffd5533804a4',
  'HTTPStatusCode': 200,
  'HTTPHeaders': {'date': 'Sun, 15 Nov 2026 23:01:40 GMT',
   'x-amz-id-2': '33af6fd6-2e3c-4223-b050-ffd5533804a4',
   'x-amz-request-id': '33af6fd6-2e3c-4223-b050-ffd5533804a4',
   'x-amzn-requestid': '33af6fd6-2e3c-4223-b050-ffd5533804a4',
   'content-length': '0'},
  'RetryAttempts': 0}}